## 2008 ITPA Threshold Power Database Analysis

In [ ]:
import numpy as np
import scipy as sp

import _simdb_common as sc

db = sc.get_db()
sims = sc.query_dataset(db, "2008")
print(f"{len(sims)} shots in 2008")

## Load scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| PLTH | `summary/global_quantities/power_loss/value` | W |
| SPLASMA | `equilibrium/time_slice/global_quantities/surface` | m^2 |
| BT | `summary/global_quantities/b0/value` | T |
| NEL | `summary/line_average/n_e/value` | m^-3 |
| PGASA (M_eff) | `summary/volume_average/meff_hydrogenic/value` | AMU |
| IP | `summary/global_quantities/ip/value` | A |
| RGEO | `summary/global_quantities/r0/value` | m |
| AMIN | `summary/boundary/minor_radius/value` | m | 
| TOK | `summary/machine` | — |
| SELEC2007 | `summary/tag/name` | — |

In [ ]:
rows = []
for i, sim in enumerate(sims, start=1):
    if i % 100 == 0 or i == len(sims):
        print(f"\r  {i}/{len(sims)} shots ({sim.alias})".ljust(80), end="", flush=True)

    md = sim.meta_dict()
    machine = md.get("machine", "")
    selec = sc.temp(md, "SELEC2007", n=0)
    n = len(selec)
    if n == 0:
        continue
    plth    = sc.path(md, "global_quantities", "power_loss", "value", n=n)
    pl      = sc.temp(md, "PL", n=n)
    splasma = sc.temp(md, "SPLASMA", "area_of_separatrix", n=n)
    bt      = np.abs(sc.path(md, "global_quantities", "b0", "value", n=n))
    nel     = sc.path(md, "line_average", "n_e", "value", n=n)
    meff    = sc.path(md, "volume_average", "meff_hydrogenic", "value", n=n)
    ip      = sc.path(md, "global_quantities", "ip", "value", n=n)
    rgeo    = sc.path(md, "boundary", "geometric_axis_r", "value", n=n)
    amin    = sc.path(md, "boundary", "minor_radius", "value", n=n)

    n_slices = min(len(a) for a in (selec, plth, pl, splasma, bt, nel, meff, ip, rgeo, amin))
    if n_slices == 0:
        continue
    sel = selec[:n_slices] == 1
    for j in range(n_slices):
        rows.append((machine, plth[j], pl[j], splasma[j], bt[j], nel[j], meff[j], ip[j],
                     rgeo[j], amin[j], bool(sel[j])))
print()

dtype = [("tok", "U16"), ("plth", float), ("pl", float), ("splasma", float), ("bt", float),
         ("nel", float), ("meff", float), ("ip", float), ("rgeo", float), ("amin", float),
         ("sel", bool)]
data = np.array(rows, dtype=dtype)

N       = len(data)
PLTH    = data["plth"]
PL      = data["pl"]
SPLASMA = data["splasma"]
BT      = data["bt"]
NEL     = data["nel"]
MEFF    = data["meff"]
IP      = data["ip"]
RGEO    = data["rgeo"]
AMIN    = data["amin"]
TOK     = data["tok"]
SEL     = data["sel"]

print("Done.")
print(f"N = {N} time-slices")
print(f"  SELEC2007=True : {np.sum(SEL)}")

## Selection Criteria

In [ ]:
mask = SEL

print(f"Total pulses        : {N}")
print(f"SELEC2007=True      : {np.sum(SEL)}")
print(f"Complete + selected : {np.sum(mask)}")
print()
print("SELEC2007=True")
for tok in np.unique(TOK[mask]):
    print(f"  {tok}: {np.sum((TOK == tok) & mask)}")

Total pulses        : 7700
SELEC2007=True      : 1024
Complete + selected : 1024

SELEC2007=True
  AUG: 175
  CMOD: 115
  D3D: 56
  JET: 562
  JFT2M: 58
  JT60U: 58


## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [ ]:
from scipy.optimize import curve_fit

m = mask

Ploss_MW = np.where(np.isnan(PLTH), PL, PLTH) / 1e6  # [MW]
ne_20    = NEL  / 1e20 # [10^20 m^-3]
ip_ma    = np.abs(IP[m])   / 1e6 # [MA]

P    = Ploss_MW[m]
Bt   = np.abs(BT[m])
ne   = ne_20[m]
Meff = MEFF[m]
S    = SPLASMA[m]
Amin = AMIN[m]
R = RGEO[m]


def model(X, C, alpha, beta, gamma):
    ne, Bt, S = X
    return C * ne**alpha * Bt**beta * S**gamma

X = (ne, Bt, S)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, beta, gamma = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  0.0500 pm  0.0014)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.689 pm 0.018)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.851  pm 0.016)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.885  pm 0.010)")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit}")

C = 0.0500 pm 0.0014   (paper  0.0500 pm  0.0014)
alpha  = 0.689 pm 0.018   (paper 0.689 pm 0.018)
beta = 0.851 pm 0.016   (paper  0.851  pm 0.016)
gamma = 0.885 pm 0.010   (paper  0.885  pm 0.010)
RMSE_fitnorm: 0.44205213034679863



## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma I_p^\delta$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [ ]:
def model(X, C, alpha, beta, gamma, delta):
    ne, Bt, S, ip_ma = X
    return C * ne**alpha * Bt**beta * S**gamma * ip_ma ** delta

X = (ne, Bt, S, ip_ma)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, beta, gamma, delta = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f}")

C = 0.0272 pm 0.0028
alpha  = 0.746 pm 0.020
beta = 1.000 pm 0.029
gamma = 1.025 pm 0.025
delta = -0.169 pm 0.028
RMSE_fitnorm: 0.444


## $P_{LH} = Cn_e^\alpha S^\gamma B_{\text{pol}}^\epsilon$, $\chi^2 = \Sigma(\frac{P_{LH} - P_{\text{scal}}}{0.15P_{LH}})^2$

In [ ]:
def model(X, C, alpha, gamma, epsilon):
    ne, S, Bpol = X
    return C * ne**alpha * S**gamma * Bpol ** epsilon

ip = ip_ma * 1e6
mu_0 = sp.constants.mu_0
Bpol = 2*np.pi * mu_0 * ip * R / S

X = (ne, S, Bpol)

popt, pcov = curve_fit(
    model, X, P,
    p0=[0.1, 0.0, 1.0, 1.0],
    sigma=0.15 * P,
    absolute_sigma=True,
)

C, alpha, gamma, epsilon = popt
perr = np.sqrt(np.diag(pcov))

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[2]:.3f}")
print(f"epsilon = {epsilon:.3f} pm {perr[3]:.3f}")

P_fit = model(X, *popt)

rmse_fit = np.sqrt(np.mean(((P - P_fit) / P_fit)**2))
print(f"RMSE_fitnorm: {rmse_fit:.3f}")

C = 0.4048 pm 0.0174
alpha  = 0.649 pm 0.020
gamma = 0.761 pm 0.012
epsilon = 0.769 pm 0.016
RMSE_fitnorm: 0.467


In [ ]:
not2 = 0

for i, meff in enumerate(Meff):
    if meff != 2: 
        not2+=1
print(f"Proportion of non-2 Meff: {not2} / {len(Meff)}")

na = 0

for i, meff in enumerate(ip_ma):
    if not(type(ip_ma[i]) == np.float64): 
        na+=1
print(f"Proportion of missing ip: {na} / {len(ip_ma)}")

Proportion of non-2 Meff: 0 / 1024
Proportion of missing ip: 0 / 1024


## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma$ (Log-space OLS)

In [ ]:
y = np.log(P)
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(Bt), np.log(S)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C     = np.exp(coef[0])
alpha = coef[1]   # ne exponent   (paper 0.717)
beta  = coef[2]   # Bt exponent   (paper 0.803)
gamma = coef[3]   # S  exponent   (paper 0.941)

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  0.0488 pm  0.057)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.717 pm 0.035)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.803  pm 0.032)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.941  pm 0.019)")
print(rms_log)



C = 0.0488 pm 0.0575   (paper  0.0488 pm  0.057)
alpha  = 0.717 pm 0.035   (paper 0.717 pm 0.035)
beta = 0.803 pm 0.032   (paper  0.803  pm 0.032)
gamma = 0.941 pm 0.019   (paper  0.941  pm 0.019)
0.30708712918570275



## $P_{LH} = Cn_e^\alpha B_t^\beta R^\gamma a^\delta$ (Log-space OLS)

In [ ]:
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(Bt), np.log(Amin), np.log(R)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C     = np.exp(coef[0])
alpha = coef[1]   # ne exponent
beta  = coef[2]   # Bt exponent
gamma = coef[3]   # R  exponent
delta = coef[4]   # a exponent

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}   (paper  2.15 pm  0.107)")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}   (paper 0.782 pm 0.037)")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}   (paper  0.772  pm 0.031)")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}   (paper  0.975  pm 0.08)")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}   (paper  0.999  pm 0.101)")


print(rms_log)

C = 2.1483 pm 0.1071   (paper  2.15 pm  0.107)
alpha  = 0.782 pm 0.037   (paper 0.782 pm 0.037)
beta = 0.772 pm 0.031   (paper  0.772  pm 0.031)
gamma = 0.975 pm 0.080   (paper  0.975  pm 0.08)
delta = 0.999 pm 0.101   (paper  0.999  pm 0.101)
0.2940859493097788


## $P_{LH} = Cn_e^\alpha B_t^\beta S^\gamma I_p^\delta$ (Log-space OLS)

In [ ]:
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(Bt), np.log(S), np.log(ip_ma)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C     = np.exp(coef[0])
alpha = coef[1]   # ne exponent
beta  = coef[2]   # Bt exponent
gamma = coef[3]   # S  exponent
delta = coef[4]   # Ip exponent

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"beta = {beta:.3f} pm {perr[2]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[3]:.3f}")
print(f"delta = {delta:.3f} pm {perr[4]:.3f}")

print(rms_log)

C = 0.0487 pm 0.2002
alpha  = 0.718 pm 0.040
beta = 0.804 pm 0.058
gamma = 0.942 pm 0.048
delta = -0.001 pm 0.055
0.30708710715031023


## $P_{LH} = Cn_e^\alpha S^\gamma B_{\text{pol}}^\epsilon$ (Log-space OLS)

In [ ]:
A = np.column_stack([np.ones_like(ne), np.log(ne), np.log(S), np.log(Bpol)])
coef, *_ = np.linalg.lstsq(A, y, rcond=None)

C       = np.exp(coef[0])
alpha   = coef[1]   # ne exponent
gamma   = coef[2]   # S  exponent
epsilon = coef[3]   # Bpol exponent

resid    = y - A @ coef
rms_log  = np.sqrt(np.mean(resid**2)) 
dof     = len(y) - A.shape[1]            # n - k degrees of freedom
sigma2  = (resid @ resid) / dof          # residual variance
cov     = sigma2 * np.linalg.inv(A.T @ A)
perr    = np.sqrt(np.diag(cov))          # standard errors on coef

print(f"C = {C:.4f} pm {perr[0]:.4f}")
print(f"alpha  = {alpha:.3f} pm {perr[1]:.3f}")
print(f"gamma = {gamma:.3f} pm {perr[2]:.3f}")
print(f"epsilon = {epsilon:.3f} pm {perr[3]:.3f}")

print(rms_log)

C = 0.3789 pm 0.0888
alpha  = 0.639 pm 0.040
gamma = 0.808 pm 0.023
epsilon = 0.758 pm 0.034
0.31857884073380277


## Correlation matrix

In [ ]:
cm = mask

S_ = SPLASMA[cm]
Bt_   = np.abs(BT[cm])
ne_   = ne_20[cm]
R_ = RGEO[cm]
ip_ = np.abs(IP[cm])
mu_0 = sp.constants.mu_0
Bpol_ = 2*np.pi * mu_0 * ip_ * R_ / S_

labels = ["ne", "Bt", "ip", "Bpol"]
data = np.vstack([ne_, Bt_, ip_, Bpol_])
corr = np.corrcoef(data)

print("Pooled correlations:\n")
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        print(f"corr({labels[i]}, {labels[j]}) = {corr[i, j]:.4f}")

Pooled correlations:

corr(ne, Bt) = 0.8035
corr(ne, ip) = -0.3494
corr(ne, Bpol) = 0.6952
corr(Bt, ip) = 0.0719
corr(Bt, Bpol) = 0.9131
corr(ip, Bpol) = 0.3388


## Per-machine correlations

In [15]:
for tok in np.unique(TOK[mask]):

    cm = mask & (tok == TOK)

    S_ = SPLASMA[cm]
    Bt_   = np.abs(BT[cm])
    ne_   = ne_20[cm]
    R_ = RGEO[cm]
    ip_ = np.abs(IP[cm])
    mu_0 = sp.constants.mu_0
    Bpol_ = 2*np.pi * mu_0 * ip_ * R_ / S_

    labels = ["ne", "Bt", "ip", "Bpol"]
    data = np.vstack([ne_, Bt_, ip_, Bpol_])
    corr = np.corrcoef(data)

    print(f"\n{tok}-only correlations:\n")
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            print(f"corr({labels[i]}, {labels[j]}) = {corr[i, j]:.4f}")


AUG-only correlations:

corr(ne, Bt) = 0.2844
corr(ne, ip) = 0.4625
corr(ne, Bpol) = 0.4730
corr(Bt, ip) = 0.6660
corr(Bt, Bpol) = 0.6694
corr(ip, Bpol) = 0.9965

CMOD-only correlations:

corr(ne, Bt) = 0.1458
corr(ne, ip) = 0.1896
corr(ne, Bpol) = 0.2124
corr(Bt, ip) = 0.5103
corr(Bt, Bpol) = 0.5160
corr(ip, Bpol) = 0.9968

D3D-only correlations:

corr(ne, Bt) = -0.1207
corr(ne, ip) = -0.0928
corr(ne, Bpol) = -0.0938
corr(Bt, ip) = 0.4794
corr(Bt, Bpol) = 0.4847
corr(ip, Bpol) = 0.9943

JET-only correlations:

corr(ne, Bt) = 0.4487
corr(ne, ip) = 0.3610
corr(ne, Bpol) = 0.4376
corr(Bt, ip) = 0.9164
corr(Bt, Bpol) = 0.9524
corr(ip, Bpol) = 0.9523

JFT2M-only correlations:

corr(ne, Bt) = 0.1601
corr(ne, ip) = 0.2974
corr(ne, Bpol) = 0.3711
corr(Bt, ip) = 0.2049
corr(Bt, Bpol) = 0.2374
corr(ip, Bpol) = 0.9898

JT60U-only correlations:

corr(ne, Bt) = 0.0382
corr(ne, ip) = 0.2100
corr(ne, Bpol) = 0.2171
corr(Bt, ip) = 0.7606
corr(Bt, Bpol) = 0.7661
corr(ip, Bpol) = 0.9972
